# TPE Optimizer (Optuna)

Joint-portfolio TPE search. See `docs/research.md` for methodology.

## Setup

In [1]:
# ruff: noqa: E402
import sys
from pathlib import Path

for _candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_candidate / "prediction_market_extensions").is_dir() and (
        _candidate / "backtests"
    ).is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from prediction_market_extensions.backtesting._notebook_support import ensure_notebook_repo_context

repo_root = ensure_notebook_repo_context()

## Configuration

In [ ]:
from decimal import Decimal

from prediction_market_extensions.backtesting._execution_config import ExecutionModelConfig
from prediction_market_extensions.backtesting._execution_config import StaticLatencyConfig
from prediction_market_extensions.backtesting._experiments import ParameterSearchExperiment
from prediction_market_extensions.backtesting._prediction_market_runner import MarketDataConfig
from prediction_market_extensions.backtesting._replay_specs import QuoteReplay
from prediction_market_extensions.backtesting.data_sources import PMXT, Polymarket, QuoteTick
from prediction_market_extensions.backtesting.optimizers import ParameterSearchConfig
from prediction_market_extensions.backtesting.optimizers import ParameterSearchWindow


NAME = "notebook_tpe_optimizer"

DATA = MarketDataConfig(
    platform=Polymarket,
    data_type=QuoteTick,
    vendor=PMXT,
    sources=(
        "local:/Volumes/LaCie/pmxt_raws",
        "archive:r2.pmxt.dev",
        "relay:209-209-10-83.sslip.io",
    ),
)

# Joint-portfolio: TPE optimizes a single parameter set evaluated across all
# markets simultaneously. PnL/fills aggregate, drawdown is computed on the
# summed equity curve, and coverage is averaged — so diversification is
# rewarded and a single stale feed does not nuke every trial.
BASE_REPLAYS = (
    QuoteReplay(market_slug="will-ludvig-aberg-win-the-2026-masters-tournament", token_index=0),
    QuoteReplay(
        market_slug="will-scottie-scheffler-win-the-2026-masters-tournament", token_index=0
    ),
    QuoteReplay(market_slug="will-jon-rahm-win-the-2026-masters-tournament", token_index=0),
)

TRAIN_WINDOWS = (
    ParameterSearchWindow(
        name="sample-a-full-window",
        start_time="2026-04-05T00:00:00Z",
        end_time="2026-04-07T23:59:59Z",
    ),
    ParameterSearchWindow(
        name="sample-b-2026-04-06-day",
        start_time="2026-04-06T00:00:00Z",
        end_time="2026-04-06T23:59:59Z",
    ),
    ParameterSearchWindow(
        name="sample-c-2026-04-07-late",
        start_time="2026-04-07T12:00:00Z",
        end_time="2026-04-07T23:59:59Z",
    ),
)

HOLDOUT_WINDOWS = (
    ParameterSearchWindow(
        name="sample-d-close-window",
        start_time="2026-04-07T00:00:00Z",
        end_time="2026-04-07T11:59:59Z",
    ),
)

STRATEGY_SPEC = {
    "strategy_path": "strategies:QuoteTickEMACrossoverStrategy",
    "config_path": "strategies:QuoteTickEMACrossoverConfig",
    "config": {
        "trade_size": Decimal("5"),
        "fast_period": "__SEARCH__:fast_period",
        "slow_period": "__SEARCH__:slow_period",
        "entry_buffer": "__SEARCH__:entry_buffer",
        "take_profit": "__SEARCH__:take_profit",
        "stop_loss": "__SEARCH__:stop_loss",
    },
}

# Continuous / log-scale space — TPE samples within these bounds.
PARAMETER_SPACE = {
    "fast_period": {"type": "int", "low": 16, "high": 128},
    "slow_period": {"type": "int", "low": 96, "high": 512},
    "entry_buffer": {"type": "float", "low": 1e-4, "high": 2e-3, "log": True},
    "take_profit": {"type": "float", "low": 2e-3, "high": 3e-2, "log": True},
    "stop_loss": {"type": "float", "low": 2e-3, "high": 3e-2, "log": True},
}

EXECUTION = ExecutionModelConfig(
    queue_position=True,
    latency_model=StaticLatencyConfig(
        base_latency_ms=75.0,
        insert_latency_ms=10.0,
        update_latency_ms=5.0,
        cancel_latency_ms=5.0,
    ),
)

MAX_TRIALS = 50
HOLDOUT_TOP_K = 5

PARAMETER_SEARCH = ParameterSearchConfig(
    name=NAME,
    data=DATA,
    base_replays=BASE_REPLAYS,
    strategy_spec=STRATEGY_SPEC,
    parameter_space=PARAMETER_SPACE,
    train_windows=TRAIN_WINDOWS,
    holdout_windows=HOLDOUT_WINDOWS,
    max_trials=MAX_TRIALS,
    random_seed=7,
    holdout_top_k=HOLDOUT_TOP_K,
    initial_cash=100.0,
    probability_window=256,
    min_quotes=500,
    min_price_range=0.005,
    min_fills_per_window=1,
    execution=EXECUTION,
    emit_html=False,
    chart_output_path="output",
    sampler="tpe",
)

EXPERIMENT = ParameterSearchExperiment(
    name=NAME,
    description="EMA TPE joint-portfolio search on PMXT L2 data",
    parameter_search=PARAMETER_SEARCH,
)

print(
    f"Configured TPE search: {MAX_TRIALS} trials over "
    f"{len(BASE_REPLAYS)} markets (joint portfolio), "
    f"{len(TRAIN_WINDOWS)} train window(s), {len(HOLDOUT_WINDOWS)} holdout window(s)."
)

## Run optimizer

In [3]:
from pprint import pprint

from prediction_market_extensions.backtesting._experiments import run_experiment_async
from prediction_market_extensions.backtesting._notebook_support import suppress_notebook_cell_output

with suppress_notebook_cell_output():
    summary = await run_experiment_async(EXPERIMENT)

print("Selected params:")
pprint(dict(summary.selected_params))
print("Leaderboard CSV:", summary.leaderboard_csv_path)
print("Summary JSON:", summary.summary_json_path)

Selected params:
{'entry_buffer': 0.0003718635067633045,
 'fast_period': 24,
 'slow_period': 421,
 'stop_loss': 0.02826408380687946,
 'take_profit': 0.014187016294426116}
Leaderboard CSV: /Users/evankolberg/prediction-market-backtesting/output/notebook_tpe_optimizer_leaderboard.csv
Summary JSON: /Users/evankolberg/prediction-market-backtesting/output/notebook_tpe_optimizer_summary.json


## Leaderboard

In [4]:
import pandas as pd
from IPython.display import display, Markdown

_leaderboard_df = pd.read_csv(summary.leaderboard_csv_path)
_display_cols = [
    c
    for c in (
        "trial_id",
        "train_median_score",
        "holdout_median_score",
        "train_median_pnl",
        "holdout_median_pnl",
        "train_median_drawdown",
        "train_median_fills",
        "params_json",
    )
    if c in _leaderboard_df.columns
]
_top = _leaderboard_df[_display_cols].head(10).copy()

_fmt = {
    "train_median_score": "{:.4f}".format,
    "holdout_median_score": "{:.4f}".format,
    "train_median_pnl": "{:.4f}".format,
    "holdout_median_pnl": "{:.4f}".format,
    "train_median_drawdown": "{:.4f}".format,
    "train_median_fills": "{:.1f}".format,
}
_fmt = {k: v for k, v in _fmt.items() if k in _top.columns}

_styled = (
    _top.style.format(_fmt, na_rep="-")
    .hide(axis="index")
    .set_caption(f"Top candidates — {EXPERIMENT.name} (TPE)")
)
display(_styled)

display(
    Markdown(
        "### Selected parameters\n\n"
        + "\n".join(f"- **{k}**: `{v}`" for k, v in dict(summary.selected_params).items())
    )
)

trial_id,train_median_score,holdout_median_score,train_median_pnl,holdout_median_pnl,train_median_drawdown,train_median_fills,params_json
1,-1000000000.0000,-1000000000.0000,0.0000,0.0000,0.0000,0.0,"{""entry_buffer"": 0.0003718635067633045, ""fast_period"": 24, ""slow_period"": 421, ""stop_loss"": 0.02826408380687946, ""take_profit"": 0.014187016294426116}"
2,-1000000000.0000,-1000000000.0000,0.0000,0.0000,0.0000,0.0,"{""entry_buffer"": 0.0001240911146748937, ""fast_period"": 76, ""slow_period"": 304, ""stop_loss"": 0.0077435023669868795, ""take_profit"": 0.00413750692578901}"
3,-1000000000.0000,-1000000000.0000,0.0000,0.0000,0.0000,0.0,"{""entry_buffer"": 0.00031305153729655524, ""fast_period"": 92, ""slow_period"": 431, ""stop_loss"": 0.004364309247234048, ""take_profit"": 0.00239098668116823}"
4,-1000000000.0000,-1000000000.0000,0.0000,0.0000,0.0000,0.0,"{""entry_buffer"": 0.0003874599660560326, ""fast_period"": 118, ""slow_period"": 184, ""stop_loss"": 0.0021395072355021765, ""take_profit"": 0.02490079745956525}"
5,-1000000000.0000,-1000000000.0000,0.0000,0.0000,0.0000,0.0,"{""entry_buffer"": 0.00019935675399995848, ""fast_period"": 83, ""slow_period"": 492, ""stop_loss"": 0.02345567474483301, ""take_profit"": 0.00883291793197674}"
6,-1000000000.0000,-,0.0000,-,0.0000,0.0,"{""entry_buffer"": 0.0009469035303373606, ""fast_period"": 31, ""slow_period"": 314, ""stop_loss"": 0.0070982296398728, ""take_profit"": 0.012241950538095943}"
7,-1000000000.0000,-,0.0000,-,0.0000,0.0,"{""entry_buffer"": 0.0003051291159941079, ""fast_period"": 39, ""slow_period"": 300, ""stop_loss"": 0.005387043471412535, ""take_profit"": 0.007286136798672624}"
8,-1000000000.0000,-,0.0000,-,0.0000,0.0,"{""entry_buffer"": 0.0002561630547236078, ""fast_period"": 110, ""slow_period"": 416, ""stop_loss"": 0.004223659200317334, ""take_profit"": 0.009429521261776949}"
9,-1000000000.0000,-,0.0000,-,0.0000,0.0,"{""entry_buffer"": 0.000716632368394708, ""fast_period"": 67, ""slow_period"": 243, ""stop_loss"": 0.006933702984315349, ""take_profit"": 0.00545251246568588}"
10,-1000000000.0000,-,0.0000,-,0.0000,0.0,"{""entry_buffer"": 0.0015110674897952736, ""fast_period"": 97, ""slow_period"": 268, ""stop_loss"": 0.01488173010872023, ""take_profit"": 0.003260300027864126}"


### Selected parameters

- **fast_period**: `24`
- **slow_period**: `421`
- **entry_buffer**: `0.0003718635067633045`
- **take_profit**: `0.014187016294426116`
- **stop_loss**: `0.02826408380687946`

## Holdout replay

In [5]:
from prediction_market_extensions.backtesting._notebook_support import (
    display_html_artifacts,
    find_updated_html_artifacts,
    snapshot_html_artifacts,
    suppress_notebook_cell_output,
)
from prediction_market_extensions.backtesting.optimizers import (
    build_parameter_search_window_backtest,
)

selected_window = HOLDOUT_WINDOWS[0] if HOLDOUT_WINDOWS else TRAIN_WINDOWS[-1]

output_root = repo_root / "output"
html_snapshot = snapshot_html_artifacts(output_root)

backtest = build_parameter_search_window_backtest(
    config=PARAMETER_SEARCH,
    window=selected_window,
    params=summary.selected_params,
    trial_id=0,
    name=f"{NAME}:{selected_window.name}:selected-params",
    emit_html=True,
    chart_output_path=str(output_root),
    return_summary_series=False,
)

with suppress_notebook_cell_output():
    results = await backtest.run_async()

updated_html = find_updated_html_artifacts(output_root, html_snapshot)
print(f"Replayed {len(results)} market(s) on window '{selected_window.name}'.")
display_html_artifacts(updated_html, repo_root=repo_root)

Replayed 3 market(s) on window 'sample-d-close-window'.


**Chart artifact:** `output/notebook_tpe_optimizer:sample-d-close-window:selected-params_will-jon-rahm-win-the-2026-masters-tournament_legacy.html`

/Users/evankolberg/prediction-market-backtesting/.venv/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
